In [ ]:
import warnings

import MDAnalysis as mda
from MDAnalysis.analysis import rms
from MDAnalysis.exceptions import SelectionError
from MDAnalysis.lib.distances import distance_array
from MDAnalysis.transformations import fit_rot_trans
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Update these paths when you have CHARMM-GUI/OpenMM output.
# WT and I825T are separate CHARMM-GUI builds, so each needs its own topology.
PSF_WT  = "../charmm-gui/wt/step3_input.psf"
PSF_MUT = "../charmm-gui/i825t/step3_input.psf"
DCD_WT  = "../simulations/wt/output.dcd"
DCD_MUT = "../simulations/i825t/output.dcd"

u_wt  = mda.Universe(PSF_WT,  DCD_WT)
u_mut = mda.Universe(PSF_MUT, DCD_MUT)
ref   = mda.Universe(PSF_WT,  DCD_WT)  # first frame of WT as reference

In [ ]:
# Superimpose on the Rpb1 backbone, then measure trigger loop (1078-1089) RMSD
# without refitting; fitting on the TL itself would hide its motion relative to Pol II.
fit_sel = "backbone and segid PROA"
tl_sel  = "backbone and segid PROA and resid 1078:1089"

R_wt  = rms.RMSD(u_wt,  ref, select=fit_sel, groupselections=[tl_sel]).run()
R_mut = rms.RMSD(u_mut, ref, select=fit_sel, groupselections=[tl_sel]).run()

# results.rmsd columns: frame, time (ps), fit-group RMSD, then one per groupselection
plt.figure(figsize=(10,4))
plt.plot(R_wt.results.rmsd[:,0],  R_wt.results.rmsd[:,3],  label="WT",    color="blue")
plt.plot(R_mut.results.rmsd[:,0], R_mut.results.rmsd[:,3], label="I825T", color="red")
plt.xlabel("Frame")
plt.ylabel("RMSD (Å)")
plt.title("Trigger Loop RMSD")
plt.legend()
plt.tight_layout()
plt.savefig("../analysis/TL_RMSD.png", dpi=150)
plt.show()

In [ ]:
# RMSF needs a trajectory already superimposed on a reference, otherwise overall
# tumbling of the complex inflates every value. Load separate copies with an
# on-the-fly fit to the WT first frame so the other cells are unaffected.
def load_aligned(psf, dcd):
    u = mda.Universe(psf, dcd)
    ref.trajectory[0]
    u.trajectory.add_transformations(
        fit_rot_trans(u.select_atoms(fit_sel), ref.select_atoms(fit_sel)))
    return u

u_wt_fit  = load_aligned(PSF_WT,  DCD_WT)
u_mut_fit = load_aligned(PSF_MUT, DCD_MUT)

# One CA per residue so the x-axis is per-residue
tl_ca = "name CA and segid PROA and resid 1078:1089"
bh_ca = "name CA and segid PROA and resid 824:844"

for label, sel in [("TriggerLoop", tl_ca), ("BridgeHelix", bh_ca)]:
    plt.figure(figsize=(10,4))
    for tag, u, color in [("WT", u_wt_fit, "blue"), ("I825T", u_mut_fit, "red")]:
        ca = u.select_atoms(sel)
        R = rms.RMSF(ca).run()
        plt.plot(ca.resids, R.results.rmsf, label=tag, color=color, marker="o")
    plt.title(f"RMSF — {label}")
    plt.xlabel("Residue (Rpb1)")
    plt.ylabel("RMSF (Å)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"../analysis/{label}_RMSF.png", dpi=150)
    plt.show()

In [ ]:
# Key GTP contact distances from Dutagaci 2023
distance_pairs = {
    # arginine has no NZ (that is lysine); take the nearer guanidinium N
    "GTP-R766":  ("resid 766 and name NH1 NH2 and segid PROA", "resname GTP and name PG"),
    "GTP-R1020": ("resid 1020 and name NH2 and segid PROA",    "resname GTP and name O1G"),
    "GTP-H1085": ("resid 1085 and name ND1 and segid PROA",    "resname GTP and name N1"),
    "GTP-L1081": ("resid 1081 and name CA  and segid PROA",    "resname GTP and name C8"),
    "GTP-Mg":    ("resname MG and name MG",                     "resname GTP and name O1A"),
}

def calc_distances(u, pairs):
    """Minimum distance (Å) between two selections at every frame, honoring PBC.

    A pair whose selection is invalid or matches no atoms is reported and
    filled with NaN, so one bad selection does not stop the others.
    """
    groups = {}
    for name, (sel1, sel2) in pairs.items():
        try:
            g1, g2 = u.select_atoms(sel1), u.select_atoms(sel2)
            if len(g1) == 0 or len(g2) == 0:
                raise ValueError(f"no atoms match '{sel1 if len(g1) == 0 else sel2}'")
            groups[name] = (g1, g2)
        except (SelectionError, ValueError) as e:
            warnings.warn(f"{name}: skipped ({e})")

    results = {name: np.full(u.trajectory.n_frames, np.nan) for name in pairs}
    for ts in u.trajectory:
        for name, (g1, g2) in groups.items():
            results[name][ts.frame] = distance_array(
                g1.positions, g2.positions, box=ts.dimensions).min()
    return results

d_wt  = calc_distances(u_wt,  distance_pairs)
d_mut = calc_distances(u_mut, distance_pairs)

fig, axes = plt.subplots(len(distance_pairs), 1, figsize=(10, 3*len(distance_pairs)))
for ax, (name, vals_wt) in zip(axes, d_wt.items()):
    ax.plot(vals_wt,          label="WT",   color="blue")
    ax.plot(d_mut[name],      label="I825T", color="red")
    ax.set_title(f"Distance: {name}")
    ax.set_ylabel("Distance (Å)")
    ax.legend()
plt.tight_layout()
plt.savefig("../analysis/GTP_distances.png", dpi=150)
plt.show()

In [ ]:
latch_pairs = {
    "R728-D760": ("resid 728 and name NH2 and segid PROA", "resid 760 and name OD1 and segid PROA"),
    "R728-L808": ("resid 728 and name NH2 and segid PROA", "resid 808 and name O   and segid PROA"),
}

d_latch_wt  = calc_distances(u_wt,  latch_pairs)
d_latch_mut = calc_distances(u_mut, latch_pairs)

for name in latch_pairs:
    plt.figure(figsize=(10,3))
    plt.plot(d_latch_wt[name],  label="WT",   color="blue")
    plt.plot(d_latch_mut[name], label="I825T", color="red")
    plt.title(f"Latch Distance: {name}")
    plt.ylabel("Distance (Å)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"../analysis/Latch_{name}.png", dpi=150)
    plt.show()
    